# Bronze: phenotype vocabulary and the synthetic clinical record

Pulls the **Human Phenotype Ontology** from its public API and generates a synthetic
paediatric cohort. No credentials, no uploads, no patient data.

| | |
| --- | --- |
| **Reference source** | `ontology.jax.org` - HPO terms for the indicator features |
| **Cohort** | Fabricated. 2,400 children, 36 months of encounters |
| **Writes** | `bronze_hpo_terms`, `bronze_patients`, `bronze_encounters`, `bronze_observations`, `bronze_family_history`, `bronze_reference_release` |

**HPO is a phenotype vocabulary, not genomic data.** It is the standard coded language
for observable clinical features - developmental delay, hypotonia, short stature. It is
used *by* genetics services, which is not the same as being genetic information. Nothing
in this pipeline reads a genome, a variant, or a test result.

Bronze stores what the source returned, verbatim, and stamps the release it read. A
phenotype term's meaning is fixed by its ontology version, and an ontology revised since
a cohort was screened is a real reproducibility problem, so the release is captured as
data rather than left implicit.

## Why the cohort is generated untidy

Real clinical records are not tidy, and a demo built on tidy data teaches the wrong
lesson. This generator deliberately produces:

* **Family history recorded inconsistently** - captured for roughly 60% of patients.
  A blank field means *nobody asked*, not *no family history*. Silver keeps that
  distinction and Gold refuses to read the blank as a negative.
* **Free-text specialty names**, with several spellings of the same service.
* **Two date formats**, because the encounter feed and the observation feed were built
  by different teams a decade apart.
* **Patients with too little record to screen**, who must be reported as *not screened*
  rather than silently counted as having no indicators.

In [ ]:
COHORT_SIZE = 2400
MONTHS_OF_HISTORY = 36
COHORT_SEED = 20260827

# Planted documentation bias. Children whose families need an interpreter have the same
# underlying rate of clustered presentation -- see the cohort cell -- but fewer of their
# features reach the record. This is what the equity check exists to find.
UNDER_DOC_INTERPRETER = 0.35
FHX_ASKED_BASE = 0.68
FHX_ASKED_INTERPRETER = 0.44
HPO_TERMS = (
    "HP:0001263,HP:0001249,HP:0000750,HP:0002376,"      # neurodevelopment
    "HP:0001252,HP:0001250,"                            # neurology
    "HP:0004322,HP:0001518,HP:0011968,"                 # growth and feeding
    "HP:0000252,HP:0000175,HP:0001999,"                 # craniofacial
    "HP:0001627,"                                       # cardiac
    "HP:0000365,HP:0000505,"                            # sensory
    "HP:0002650"                                        # skeletal
)
PIPELINE_RUN_ID = ""# ---- clinical notes ----------------------------------------------------------
# Share of the findings that under-documentation removed from the coded feed which
# still get described in prose. This is the whole point of reading notes: the finding
# was observed and written down, it just never reached a coded field.
NOTE_RESCUE_SHARE = 0.62
# Interpreter-needing families lose some of that too -- a shorter consultation produces
# a shorter note. Extraction narrows the gap; it does not close it.
NOTE_RESCUE_INTERPRETER = 0.41
NOTES_PER_ENCOUNTER = 0.55
# Share of mentions written in shorthand the extractor's lexicon does not carry.
# Without this, notes generated from a vocabulary and read by the same vocabulary
# score 100% precision and recall -- a measure of the plumbing, not of anything
# clinical. Real charts do not agree with a dictionary, and the demo should not
# pretend otherwise.
UNSEEN_PHRASING_SHARE = 0.18

In [ ]:
import json
import random
import urllib.parse
import urllib.request
from datetime import date, timedelta

from pyspark.sql import functions as F
from pyspark.sql.types import (BooleanType, DateType, IntegerType, StringType,
                               StructField, StructType)

RUN_ID = PIPELINE_RUN_ID or f"local-{date.today():%Y%m%d}"
RNG = random.Random(COHORT_SEED)
TODAY = date(2026, 8, 27)
WINDOW_START = TODAY - timedelta(days=30 * MONTHS_OF_HISTORY)

print(f"run id        {RUN_ID}")
print(f"cohort        {COHORT_SIZE} synthetic patients")
print(f"window        {WINDOW_START} .. {TODAY}")

In [ ]:
# ---------------------------------------------------------------- HPO terms
# The public HPO API. Terms are fetched rather than hard-coded so the label and
# definition in the evidence contract are the ontology's words, not ours -- an agent
# citing "HP:0001263" should be citing something a clinician can look up.

HPO_API = "https://ontology.jax.org/api/hp/terms/"
requested = [t.strip() for t in HPO_TERMS.split(",") if t.strip()]
records, failures = [], []

for term_id in requested:
    try:
        request = urllib.request.Request(
            HPO_API + urllib.parse.quote(term_id, safe=""),
            headers={"Accept": "application/json", "User-Agent": "fabric-demo/1.0"})
        with urllib.request.urlopen(request, timeout=45) as response:
            payload = json.loads(response.read().decode("utf-8"))
        records.append({
            "hpo_id": term_id,
            "name": payload.get("name"),
            "definition": payload.get("definition"),
            "raw_json": json.dumps(payload)[:8000],
            "fetched_run_id": RUN_ID,
        })
    except Exception as exc:
        failures.append((term_id, f"{type(exc).__name__}: {exc}"))

print(f"fetched {len(records)} / {len(requested)} terms")
for term_id, reason in failures:
    print(f"  MISSED {term_id}: {reason}")

# A term we could not fetch must not silently become a term we invented. Record the
# failure as data so Silver's gate can see it.
for term_id, reason in failures:
    records.append({"hpo_id": term_id, "name": None, "definition": None,
                    "raw_json": json.dumps({"error": reason}),
                    "fetched_run_id": RUN_ID})

hpo_schema = StructType([
    StructField("hpo_id", StringType()),
    StructField("name", StringType()),
    StructField("definition", StringType()),
    StructField("raw_json", StringType()),
    StructField("fetched_run_id", StringType()),
])
spark.createDataFrame(records, hpo_schema).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("bronze_hpo_terms")
print("wrote bronze_hpo_terms")

In [ ]:
# ------------------------------------------------------- the release stamp
release = [{
    "source": "Human Phenotype Ontology",
    "endpoint": HPO_API,
    "read_on": TODAY,
    "terms_requested": len(requested),
    "terms_returned": len(requested) - len(failures),
    "run_id": RUN_ID,
}]
release_schema = StructType([
    StructField("source", StringType()),
    StructField("endpoint", StringType()),
    StructField("read_on", DateType()),
    StructField("terms_requested", IntegerType()),
    StructField("terms_returned", IntegerType()),
    StructField("run_id", StringType()),
])
spark.createDataFrame(release, release_schema).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("bronze_reference_release")
print("wrote bronze_reference_release")

In [ ]:
# ------------------------------------------------ the synthetic clinical record
# Several spellings of the same services, because that is what a real extract holds.
SPECIALTIES = [
    "General Paediatrics", "General Pediatrics", "Neurology", "Neurology  ",
    "Cardiology", "Cardiology (Outpatient)", "Ophthalmology", "ENT",
    "Otolaryngology", "Developmental Paediatrics", "Developmental Pediatrics",
    "Respirology", "Gastroenterology", "Endocrinology", "Nephrology",
    "Orthopaedics", "Orthopedics", "Metabolics", "Immunology", "Dermatology",
]

# Grouped by the body system each belongs to. Features spread across several systems
# is the strongest single reason a paediatrician refers to genetics, so the generator
# has to be able to produce that pattern -- and, for most children, not produce it.
INDICATOR_TERMS = [
    "HP:0001263", "HP:0001249", "HP:0000750", "HP:0002376",
    "HP:0001252", "HP:0001250",
    "HP:0004322", "HP:0001518", "HP:0011968",
    "HP:0000252", "HP:0000175", "HP:0001999",
    "HP:0001627",
    "HP:0000365", "HP:0000505",
    "HP:0002650",
]

# Equity attributes. These exist so the pipeline can measure whether the flag lands
# evenly -- NOT so they can be used as features. Gold asserts they never reach scoring.
LANGUAGES = ["English", "English", "English", "English", "Mandarin", "Cantonese",
             "Tamil", "Urdu", "Spanish", "Arabic", "Tagalog", "Portuguese",
             "Somali", "Farsi", "Gujarati"]

patients, encounters, observations, family_history = [], [], [], []

for index in range(COHORT_SIZE):
    patient_id = f"SYN-{index + 1:05d}"
    birth = TODAY - timedelta(days=RNG.randint(200, 17 * 365))
    language = RNG.choice(LANGUAGES)
    interpreter = language != "English" and RNG.random() < 0.72

    # A minority of the cohort genuinely has a clustered presentation. The generator
    # knows which; the pipeline must not, and never reads this column.
    latent = RNG.random() < 0.11

    enrolled = WINDOW_START + timedelta(days=RNG.randint(0, 30 * MONTHS_OF_HISTORY))

    patients.append({
        "patient_id": patient_id, "birth_date": birth, "enrolled_on": enrolled,
        "primary_language": language, "interpreter_required": interpreter,
        "_latent_cluster": latent, "run_id": RUN_ID,
    })

    visit_count = RNG.randint(6, 22) if latent else RNG.randint(1, 11)
    for _ in range(visit_count):
        when = enrolled + timedelta(days=RNG.randint(0, max(1, (TODAY - enrolled).days)))
        encounters.append({
            "encounter_id": f"ENC-{len(encounters) + 1:07d}",
            "patient_id": patient_id,
            # The encounter feed writes dd/MM/yyyy.
            "encounter_date_raw": when.strftime("%d/%m/%Y"),
            "specialty_raw": RNG.choice(SPECIALTIES),
            "admitted": RNG.random() < (0.22 if latent else 0.07),
            "diagnosis_recorded": RNG.random() > (0.55 if latent else 0.2),
            "run_id": RUN_ID,
        })

    feature_count = (RNG.randint(2, 6) if latent
                     else RNG.choices([0, 1, 2], weights=[62, 26, 12])[0])
    drawn = RNG.sample(INDICATOR_TERMS, min(feature_count, len(INDICATOR_TERMS)))

    # Under-documentation, not under-prevalence. `latent` above was drawn before
    # language was consulted, so the two groups carry clustered presentation at the
    # same rate. What differs is how much of it gets written down: consultations run
    # shorter through an interpreter, history-taking is harder, and description is less
    # likely to reach a coded field. The pipeline can only ever see the record.
    drawn = [t for t in drawn
             if not (interpreter and RNG.random() < UNDER_DOC_INTERPRETER)]

    for term in drawn:
        when = enrolled + timedelta(days=RNG.randint(0, max(1, (TODAY - enrolled).days)))
        observations.append({
            "observation_id": f"OBS-{len(observations) + 1:07d}",
            "patient_id": patient_id,
            # The observation feed writes MM/dd/yyyy. Same estate, different decade.
            "observed_date_raw": when.strftime("%m/%d/%Y"),
            "hpo_id": term,
            "recorded_by_raw": RNG.choice(
                ["Clinician", "clinician", "CLINICIAN", "Nurse Practitioner", "NP"]),
            "run_id": RUN_ID,
        })

    # Family history is asked inconsistently, and least often where it takes longest
    # to ask. A missing row is a missing question, not a negative answer.
    if RNG.random() < (FHX_ASKED_INTERPRETER if interpreter else FHX_ASKED_BASE):
        family_history.append({
            "patient_id": patient_id,
            "affected_first_degree": ((RNG.random() < 0.35) if latent
                                      else (RNG.random() < 0.06)),
            "consanguinity": ((RNG.random() < 0.16) if latent
                              else (RNG.random() < 0.03)),
            "recurrent_pregnancy_loss": ((RNG.random() < 0.18) if latent
                                         else (RNG.random() < 0.05)),
            "asked_on_raw": (enrolled + timedelta(days=RNG.randint(0, 200)))
                            .strftime("%d/%m/%Y"),
            "run_id": RUN_ID,
        })

asked_share = len(family_history) / len(patients)
print(f"patients        {len(patients):>7,}")
print(f"encounters      {len(encounters):>7,}")
print(f"observations    {len(observations):>7,}")
print(f"family history  {len(family_history):>7,}  ({asked_share:.0%} of the cohort "
      f"-- the rest were never asked)")

In [ ]:
# ------------------------------------------------------------------ persist
patient_schema = StructType([
    StructField("patient_id", StringType()),
    StructField("birth_date", DateType()),
    StructField("enrolled_on", DateType()),
    StructField("primary_language", StringType()),
    StructField("interpreter_required", BooleanType()),
    StructField("_latent_cluster", BooleanType()),
    StructField("run_id", StringType()),
])
encounter_schema = StructType([
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("encounter_date_raw", StringType()),
    StructField("specialty_raw", StringType()),
    StructField("admitted", BooleanType()),
    StructField("diagnosis_recorded", BooleanType()),
    StructField("run_id", StringType()),
])
observation_schema = StructType([
    StructField("observation_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("observed_date_raw", StringType()),
    StructField("hpo_id", StringType()),
    StructField("recorded_by_raw", StringType()),
    StructField("run_id", StringType()),
])
family_schema = StructType([
    StructField("patient_id", StringType()),
    StructField("affected_first_degree", BooleanType()),
    StructField("consanguinity", BooleanType()),
    StructField("recurrent_pregnancy_loss", BooleanType()),
    StructField("asked_on_raw", StringType()),
    StructField("run_id", StringType()),
])

for rows, schema, table in [
    (patients, patient_schema, "bronze_patients"),
    (encounters, encounter_schema, "bronze_encounters"),
    (observations, observation_schema, "bronze_observations"),
    (family_history, family_schema, "bronze_family_history"),
]:
    spark.createDataFrame(rows, schema).write.mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(table)
    print(f"wrote {table:26} {len(rows):>7,} rows")

## Free-text clinical notes

The coded feed above is what a structured pipeline sees. These are the notes it cannot read -- and a share of the findings under-documentation removed from the coded feed are described here in prose and nowhere else.

In [ ]:
# --------------------------------------------------- free-text clinical notes
# Drawn from a SEPARATE RNG stream. bronze_observations above must stay byte-identical
# so the coded-only figures in the recording remain a valid baseline to measure against.

# Natural phrasing per HPO term. A note says "low tone", never "HP:0001252" -- which is
# exactly why a coded-only pipeline cannot see it.
PHRASING = {
    "HP:0001263": ["global developmental delay", "developmental delay",
                   "delayed milestones", "delay across all developmental domains"],
    "HP:0001249": ["intellectual disability", "cognitive impairment",
                   "significant learning difficulties"],
    "HP:0000750": ["speech delay", "delayed speech and language",
                   "very limited expressive language"],
    "HP:0002376": ["developmental regression", "loss of previously acquired skills",
                   "regression of milestones"],
    "HP:0001252": ["hypotonia", "low tone", "reduced muscle tone", "a floppy infant"],
    "HP:0001250": ["seizures", "seizure activity", "convulsive episodes"],
    "HP:0004322": ["short stature", "height below the third centile",
                   "growth restriction"],
    "HP:0001518": ["small for gestational age", "low birth weight for dates"],
    "HP:0011968": ["feeding difficulties", "poor feeding", "ongoing difficulty feeding"],
    "HP:0000252": ["microcephaly", "a small head circumference",
                   "OFC below the second centile"],
    "HP:0000175": ["cleft palate", "a palatal cleft"],
    "HP:0001999": ["dysmorphic features", "facial dysmorphism",
                   "an unusual facial appearance"],
    "HP:0001627": ["a structural cardiac anomaly", "abnormal heart morphology",
                   "a congenital heart defect"],
    "HP:0000365": ["hearing impairment", "sensorineural hearing loss", "hearing loss"],
    "HP:0000505": ["visual impairment", "reduced visual acuity", "poor vision"],
    "HP:0002650": ["scoliosis", "curvature of the spine", "a scoliotic curve"],
}

# Clinical shorthand the extractor is NOT given. Every one of these is something a
# paediatrician would actually write, and none contains a lexicon surface form, so a
# mention phrased this way is invisible to extraction. This is what makes the recall
# figure mean something.
UNSEEN_PHRASING = {
    "HP:0001263": ["GDD", "globally delayed"],
    "HP:0001249": ["below-average cognitive function"],
    "HP:0000750": ["not yet using words", "non-verbal"],
    "HP:0002376": ["skills have gone backwards", "lost skills previously gained"],
    "HP:0001252": ["hypotonic", "floppy"],
    "HP:0001250": ["fitting episodes", "epileptic events"],
    "HP:0004322": ["faltering growth", "below the 3rd centile for height"],
    "HP:0001518": ["SGA", "birth weight below expected"],
    "HP:0011968": ["failure to thrive", "struggles with feeds"],
    "HP:0000252": ["small OFC", "head circumference well below average"],
    "HP:0000175": ["repaired cleft", "roof of the mouth not fused"],
    "HP:0001999": ["unusual features", "distinctive facies"],
    "HP:0001627": ["CHD", "murmur with structural findings on echo"],
    "HP:0000365": ["hard of hearing", "fails hearing screens"],
    "HP:0000505": ["vision problems", "does not fix and follow"],
    "HP:0002650": ["curved spine", "spinal asymmetry"],
}

UNSEEN_TEMPLATES = [
    "Impression: {p}.",
    "{P} on review.",
    "Documented this visit: {p}.",
    "Assessment: {p}.",
]
PRESENT_TEMPLATES = [
    "On examination there is {p}.",
    "Mother reports {p}.",
    "Ongoing concerns regarding {p}.",
    "{P} noted, unchanged since the last review.",
    "Persistent {p}.",
    "Parents describe {p} over the past year.",
]
# Distractors. A dictionary match alone would read these as findings, which is why the
# extractor has to carry an assertion and not just a term id.
NEGATED_TEMPLATES = [
    "No evidence of {p}.",
    "Examination negative for {p}.",
    "Considered and ruled out: {p}.",
    "Parents deny {p}.",
]
FAMILY_TEMPLATES = [
    "Family history of {p} in a maternal cousin.",
    "Mother has a history of {p}.",
    "Father reports {p} in childhood.",
    "An older sibling was investigated for {p}.",
]
OPENERS = [
    "{spec} clinic review.",
    "Seen in {spec} today.",
    "{spec} follow-up appointment.",
    "Reviewed in {spec}.",
]
CLOSERS = [
    "Plan: review in six months.",
    "Plan: continue current management, review as needed.",
    "To be discussed at the next multidisciplinary meeting.",
    "Bloods requested; parents to contact us with any concerns.",
    "Plan: routine follow-up.",
]

notes, note_findings = [], []
_note_rng_base = COHORT_SEED * 7919

# Rebuild the per-patient view we need. patients/encounters/observations are already
# generated above; this walks them rather than regenerating anything.
coded_by_patient = {}
for obs in observations:
    coded_by_patient.setdefault(obs["patient_id"], set()).add(obs["hpo_id"])

enc_by_patient = {}
for enc in encounters:
    enc_by_patient.setdefault(enc["patient_id"], []).append(enc)

for index, patient in enumerate(patients):
    pid = patient["patient_id"]
    nrng = random.Random(_note_rng_base + index)
    interpreter = patient["interpreter_required"]
    coded = coded_by_patient.get(pid, set())

    # Findings this patient has that never reached the coded feed. This is an
    # independent draw calibrated to the same shape rather than literally the terms
    # the loop above discarded -- recovering those would mean reordering that RNG
    # stream, and bronze_observations has to stay byte-identical so the coded-only
    # figures in the recording remain a valid baseline.
    truth_rng = random.Random(_note_rng_base + 31 * index + 7)
    dropped = [t for t in INDICATOR_TERMS if t not in coded
               and truth_rng.random() < (0.16 if patient["_latent_cluster"] else 0.02)]

    rescue = NOTE_RESCUE_INTERPRETER if interpreter else NOTE_RESCUE_SHARE
    note_only = [t for t in dropped if nrng.random() < rescue]

    # Everything the notes may mention as present: coded findings get re-described in
    # prose (as they would be), note-only findings appear nowhere else.
    mentionable = [(t, "coded") for t in sorted(coded)] + \
                  [(t, "note_only") for t in note_only]
    if not mentionable and nrng.random() < 0.5:
        continue

    patient_encounters = enc_by_patient.get(pid, [])
    nrng.shuffle(patient_encounters)
    note_count = max(1, int(len(patient_encounters) * NOTES_PER_ENCOUNTER))
    # A shorter consultation produces a shorter note.
    if interpreter:
        note_count = max(1, int(note_count * 0.7))

    assigned = {}
    for slot, (term, origin) in enumerate(mentionable):
        assigned.setdefault(slot % max(note_count, 1), []).append((term, origin))

    for n in range(note_count):
        enc = patient_encounters[n % len(patient_encounters)] if patient_encounters \
            else None
        note_id = f"NOTE-{len(notes) + 1:07d}"
        spec = (enc["specialty_raw"].strip() if enc else "General Paediatrics")
        sentences = [nrng.choice(OPENERS).format(spec=spec)]

        for term, origin in assigned.get(n, []):
            unseen = nrng.random() < UNSEEN_PHRASING_SHARE
            phrase = nrng.choice(
                UNSEEN_PHRASING[term] if unseen else PHRASING[term])
            template = nrng.choice(
                UNSEEN_TEMPLATES if unseen else PRESENT_TEMPLATES)
            sentence = template.format(p=phrase, P=phrase[0].upper() + phrase[1:])
            sentences.append(sentence)
            note_findings.append({
                "note_id": note_id, "patient_id": pid, "hpo_id": term,
                "assertion": "present", "origin": origin,
                "phrase_used": phrase, "phrasing_in_lexicon": not unseen,
                "run_id": RUN_ID,
            })

        # Distractors, so the extractor has something to get wrong.
        if nrng.random() < 0.34:
            term = nrng.choice(INDICATOR_TERMS)
            phrase = nrng.choice(PHRASING[term])
            sentences.append(nrng.choice(NEGATED_TEMPLATES).format(
                p=phrase, P=phrase[0].upper() + phrase[1:]))
            note_findings.append({
                "note_id": note_id, "patient_id": pid, "hpo_id": term,
                "assertion": "negated", "origin": "distractor",
                "phrase_used": phrase, "phrasing_in_lexicon": True,
                "run_id": RUN_ID})
        if nrng.random() < 0.22:
            term = nrng.choice(INDICATOR_TERMS)
            phrase = nrng.choice(PHRASING[term])
            sentences.append(nrng.choice(FAMILY_TEMPLATES).format(
                p=phrase, P=phrase[0].upper() + phrase[1:]))
            note_findings.append({
                "note_id": note_id, "patient_id": pid, "hpo_id": term,
                "assertion": "family_history", "origin": "distractor",
                "phrase_used": phrase, "phrasing_in_lexicon": True,
                "run_id": RUN_ID})

        sentences.append(nrng.choice(CLOSERS))
        when = enc["encounter_date_raw"] if enc else None
        notes.append({
            "note_id": note_id, "patient_id": pid,
            "encounter_id": enc["encounter_id"] if enc else None,
            "note_date_raw": when,
            "author_specialty_raw": spec,
            "note_text": " ".join(sentences),
            "run_id": RUN_ID,
        })

only_in_prose = sum(1 for f in note_findings if f["origin"] == "note_only")
print(f"notes           {len(notes):>7,}")
unseen_share = sum(1 for f in note_findings if not f["phrasing_in_lexicon"])
print(f"note findings   {len(note_findings):>7,}  "
      f"({only_in_prose:,} present ONLY in prose, never coded)")
print(f"  of which {unseen_share:,} use shorthand the extractor's lexicon does not hold")

In [ ]:
# ------------------------------------------------------------ persist the notes
note_schema = StructType([
    StructField("note_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("encounter_id", StringType()),
    StructField("note_date_raw", StringType()),
    StructField("author_specialty_raw", StringType()),
    StructField("note_text", StringType()),
    StructField("run_id", StringType()),
])
# The answer key for the notes. Silver's extractor never reads this; validation does,
# to measure what extraction found and what it missed.
truth_schema = StructType([
    StructField("note_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("hpo_id", StringType()),
    StructField("assertion", StringType()),
    StructField("origin", StringType()),
    StructField("phrase_used", StringType()),
    StructField("phrasing_in_lexicon", BooleanType()),
    StructField("run_id", StringType()),
])
for rows, schema, table in [
    (notes, note_schema, "bronze_clinical_notes"),
    (note_findings, truth_schema, "_bronze_note_truth"),
]:
    spark.createDataFrame(rows, schema).write.mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(table)
    print(f"wrote {table:26} {len(rows):>7,} rows")

In [ ]:
# --------------------------------------------------------------- sanity check
for table in ["bronze_hpo_terms", "bronze_patients", "bronze_encounters",
              "bronze_observations", "bronze_family_history",
              "bronze_reference_release"]:
    print(f"{table:26} {spark.table(table).count():>7,}")

named = spark.table("bronze_hpo_terms").filter(F.col("name").isNotNull()).count()
total = spark.table("bronze_hpo_terms").count()
print(f"\nHPO terms carrying a label: {named}/{total}")
if named < total * 0.8:
    raise ValueError(
        f"only {named}/{total} HPO terms resolved. The evidence contract cites term "
        f"labels, so proceeding would mean citing labels we never fetched.")
print("bronze complete")